In [57]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from kilosort import io, run_kilosort
import kilosort
from kilosort.data_tools import get_best_channels
import gc
import torch


In [58]:
gc.collect()

21952

In [59]:
torch.cuda.empty_cache()

In [60]:
def makeProbeFromFile(chanMapDict, spacing=(1,1), grouped=0, chnOffset =0):
    chanMap = pd.read_excel(chanMapDict['filename'],sheet_name = chanMapDict['sheet']).iloc[chanMapDict['cellIdx'][0], chanMapDict['cellIdx'][1]]
    chanMap = np.rot90(np.flipud(chanMap),-1)
    [xc, yc] = [np.where(pd.notna(chanMap))[1], np.where(pd.notna(chanMap))[0]]
    
    xc = xc*spacing[0]
    yc = yc*spacing[1]
    
    n_chan = len(xc)
    connected = np.ones((n_chan,1)).astype('bool')
    
    chanMap = np.arange(n_chan) + chnOffset
    
    if grouped:
        kcoords = np.ones(n_chan)
    else:
        kcoords = np.arange(n_chan)

    probe = {
        'chanMap': chanMap,
        'xc': xc,
        'yc': yc,
        'kcoords': kcoords,
        'n_chan': n_chan
    }
    return probe

def makeProbeFromParameters(n_chan, spacing, grouped=0, chnOffset= 0, x0=0, y0=0):
    chanMap = np.arange(n_chan) + chnOffset
    xc = np.array([x0 + spacing[0]*i for i in reversed(range(n_chan))])
    yc = np.array([y0 + spacing[1]*i for i in reversed(range(n_chan))]) # reversed makes things identical to how they were in matlab
    
    if grouped:
        kcoords = np.ones(n_chan)
    else:
        kcoords = np.arange(n_chan)
    
    
    probe = {
        'chanMap': chanMap,
        'xc': xc,
        'yc': yc,
        'kcoords': kcoords,
        'n_chan': n_chan
    }
    return probe
    


In [61]:
def getProbeSubset(probe, channels):
    keys = ['chanMap', 'xc', 'yc', 'kcoords']
    probeSubset = {k: probe[k][channels] for k in keys}
    probeSubset['n_chan'] = len(channels)
    return probeSubset

In [62]:
# #chanMapPath = processingPath / "chanMapFiles" / monkey_name

# # ONLY FOR UTAH ARRAYS
# chanMapDicts = [{'filename': chanMapPath / 'SN 7623-000080 Mapping and Impedance' / 'SN 7623-000080.xlsm',
#                 'sheet': 'Z check',
#                 'cellIdx': [np.arange(14, 25), np.arange(15,25)]},
#                {'filename': chanMapPath / 'SN 7624-000191 Mapping and Impedance' /'1125-15 SN 7624-000191.xlsm',
#                'sheet': 'Z check',
#                'cellIdx':[np.arange(14, 25), np.arange(15,25)]}, {}]


In [63]:

# load probe info or make it if it doesnt exist


# arrayDict = {'labels': ['UT1', 'UT2', 'lam'],
#             'n_chan': [(), (), 64],
#             'spacing': [(400,400), (400,400), (0,50)],
#              'grouped': [1,1,1],
#             'chanMapDict': chanMapDicts}


labels = ['lam']
n_chan = [64]
spacing = [(0,50)]
grouped = [1]
chanMapDict = []


arrayDict = {'labels': labels,
            'n_chan': n_chan,
            'spacing': spacing,
            'grouped': grouped,
            'chanMapDict': []}

# for each channel, determine if it has a channel map and if not specifiy geometry
probeDicts = []
chnOffset = 0

for i in range(len(arrayDict['labels'])):
    # if this array has an excel file, make probe info from file
    if len(arrayDict['chanMapDict']) > 0:
        chanMapDict = arrayDict['chanMapDict'][i]
    else:
        chanMapDict = []
    
    if chanMapDict:
        probeDicts.append(makeProbeFromFile(chanMapDict=chanMapDict,
                                            spacing=arrayDict['spacing'][i],
                                            grouped=arrayDict['grouped'][i],
                                            chnOffset=chnOffset)) 
        chnOffset = chnOffset + probeDicts[i]['n_chan']
    else:
        probeDicts.append(makeProbeFromParameters(n_chan=arrayDict['n_chan'][i],
                                                  spacing=arrayDict['spacing'][i],
                                                  grouped=arrayDict['grouped'][i],
                                                  chnOffset=chnOffset))
        chnOffset = chnOffset + probeDicts[i]['n_chan']
            

In [64]:
probeDicts

[{'chanMap': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
         17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
         34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
         51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
  'xc': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
  'yc': array([3150, 3100, 3050, 3000, 2950, 2900, 2850, 2800, 2750, 2700, 2650,
         2600, 2550, 2500, 2450, 2400, 2350, 2300, 2250, 2200, 2150, 2100,
         2050, 2000, 1950, 1900, 1850, 1800, 1750, 1700, 1650, 1600, 1550,
         1500, 1450, 1400, 1350, 1300, 1250, 1200, 1150, 1100, 1050, 1000,
          950,  900,  850,  800,  750,  700,  650,  600,  550,  500,  450,
          400,  350,  300,  250,  200,  150,  100,   50,    0]),
  'kcoords': array([1.

In [65]:
dataPath = Path.home() / "Data" / "V1_Fovea"

SAVE_PATH = dataPath / "Sprout" /"260508" / "260508_152524_Sprout" / "kilosort_laminar_1to64" 
print(SAVE_PATH)

/home/clab/Data/V1_Fovea/Sprout/260508/260508_152524_Sprout/kilosort_laminar_1to64


In [66]:
probeSubset = getProbeSubset(probeDicts[0], range(64))
probeSubset

{'chanMap': array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
        51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63]),
 'xc': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'yc': array([3150, 3100, 3050, 3000, 2950, 2900, 2850, 2800, 2750, 2700, 2650,
        2600, 2550, 2500, 2450, 2400, 2350, 2300, 2250, 2200, 2150, 2100,
        2050, 2000, 1950, 1900, 1850, 1800, 1750, 1700, 1650, 1600, 1550,
        1500, 1450, 1400, 1350, 1300, 1250, 1200, 1150, 1100, 1050, 1000,
         950,  900,  850,  800,  750,  700,  650,  600,  550,  500,  450,
         400,  350,  300,  250,  200,  150,  100,   50,    0]),
 'kcoords': array([1., 1., 1., 1., 

In [68]:
fs = 40000.0
highpass_cutoff = 300.0
Th_universal = 11.0
Th_learned = 8.0
nt = 81
#n_pcs = 3

# for utah arrays
n_chan_bin = 64

#nblocks = 0 # 0 for Utah
x_centers = None # 4for Utah

settings = {
    "filename": SAVE_PATH,
    "n_chan_bin": n_chan_bin,
    "fs": fs,
    "highpass_cutoff": highpass_cutoff,
    "batch_size": int(2*fs),   # 2.5 for 32 chan   
    "Th_universal": Th_universal,
    "Th_learned": Th_learned,
    "nt": nt,
    "nearest_templates": n_chan_bin
#     "x_centers": x_centers,
#     "nearest_templates": 32,
#     "dmin": dmin
}

# settings["x_centers"] = np.linspace(
#     np.min(probeSubset["xc"]),
#     np.max(probeSubset["xc"]),
#     4  
# )

ops, st, clu, tF, Wall, similar_templates, is_ref, est_contam_rate, kept_spikes = run_kilosort(
    settings=settings,
    probe=probeSubset,
    filename= SAVE_PATH / '260508_152524_Sprout_laminar_1to64.dat',
    results_dir = SAVE_PATH,
    data_dtype="int16",
    do_CAR=True,
    invert_sign=False,
    save_preprocessed_copy=True,
    clear_cache=True
)


kilosort.run_kilosort: Kilosort version 4.1.7
kilosort.run_kilosort: Python version 3.11.14
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: System information:
kilosort.run_kilosort: Linux-6.8.0-117-generic-x86_64-with-glibc2.35 x86_64
kilosort.run_kilosort: x86_64
kilosort.run_kilosort: Using GPU for PyTorch computations. Specify `device` to change this.
kilosort.run_kilosort: Using CUDA device: NVIDIA GeForce RTX 5070 11.48GB
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: Sorting [PosixPath('/home/clab/Data/V1_Fovea/Sprout/260508/260508_152524_Sprout/kilosort_laminar_1to64/260508_152524_Sprout_laminar_1to64.dat')]
kilosort.run_kilosort: clear_cache=True
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage before sorting
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     1.90 %
kilosort.run_kilosort: Mem used:     96.70 %     |    

### 